### **LAB — Data Lakehouse médical**
### PostgreSQL · Apache Spark / PySpark · Apache Iceberg · RustFS · Jupyter

**Sujet :** ingestion multi-sources, couches Bronze / Silver / Gold et tables transactionnelles.  
**Environnement :** infrastructure Docker fournie et configurée.  
**Sources :** jeu de données médicales entièrement synthétiques Synthea .


---

**Sommaire**  
- **Partie 0** — Vérifier l'infrastructure, télécharger et préparer Synthea  
- **Partie 1** — Bronze : ingestion des trois sources dans Apache Iceberg  
- **Partie 2** — Silver : nettoyage, contrôles et Quarantine  
- **Partie 3** — Gold : création d'indicateurs analytiques  
- **Validation finale** — Snapshots Iceberg, réingestion, livrables

## Contexte  

Un établissement de santé dispose de plusieurs systèmes qui n'exposent pas leurs données de la même façon :

| Source simulée | Donnée utilisée | Support dans ce TP |
|---|---|---|
| Système de gestion des patients | Données démographiques | Table relationnelle **PostgreSQL** `patients_source` |
| Logiciel de gestion des prises en charge | Rencontres / consultations | Fichier **CSV** `encounters.csv` |
| Système clinique et laboratoire | Observations, examens et résultats | Fichier **JSON Lines** `observations.jsonl` |

L'établissement veut centraliser ces informations et produire des tableaux de bord d'activité. Les données devront être traçables, contrôlées et réutilisables.

**Objectif du Lakehouse :** conserver les données de diverses origines, les transformer à l'aide de Spark, les gérer en tables Apache Iceberg et fournir des tables analytiques fiables.

## Infrastructure fournie :

L'environnement est exécuté dans des **conteneurs Docker**. Vous n'installez pas PostgreSQL, Spark, Iceberg ou RustFS manuellement : ces composants sont déjà disponibles dans les images et la configuration fournies.

| Service | Rôle | Accès depuis Windows / navigateur | Accès entre conteneurs Docker |
|---|---|---|---|
| **PostgreSQL** | Base source des patients | `localhost:5433` | `postgres:5432` |
| **RustFS** | Stockage objet compatible S3 | Console : [http://localhost:9001](http://localhost:9001) ; API : `localhost:9000` | `rustfs:9000` |
| **Iceberg REST** | Catalogue de tables | [http://localhost:8181/v1/config](http://localhost:8181/v1/config) | `iceberg-rest:8181` |
| **Spark + Jupyter** | Moteur de calcul et notebooks | [http://localhost:8888](http://localhost:8888) | Service `spark-jupyter` |

### Accès:

| Service | Identifiants à utiliser |
|---|---|
| PostgreSQL | **Utilisateur :** `user` · **Mot de passe :** `0000` · **Base :** `hospital` |
| Jupyter | **Token :** `medical-lab-local` |
| RustFS | Clé d'accès et clé secrète `RUSTFS_ACCESS_KEY` : user et  `RUSTFS_SECRET_KEY`: 0000|
| Iceberg REST | Dans ce lab, l'API locale ne demande pas de connexion utilisateur distincte |

### Comprendre les rôles avant de démarrer

```text
PostgreSQL (patients)    CSV (encounters)    JSONL (observations)
         \                  |                     /
          \                 |                    /
                   Apache Spark
                        |
               Apache Iceberg (tables)
                        |
         Catalogue REST (références aux tables)
                        |
          RustFS medical-warehouse (fichiers)
                        |
             BRONZE -> SILVER -> GOLD
```

- **Spark** : lit, transforme, contrôle et écrit les données.
- **Parquet** : format physique des fichiers de données analytiques.
- **Apache Iceberg** : format de table avec métadonnées, transactions et snapshots.
- **Catalogue REST** : permet de retrouver les tables Iceberg ; il ne calcule pas les requêtes.
- **RustFS** : conserve les objets, y compris les fichiers Parquet et les métadonnées Iceberg.
- **Medallion** : organisation logique en Bronze (données ingérées), Silver (fiables) et Gold (analytiques).

### Deux buckets RustFS

| Bucket | Usage |
|---|---|
| `medical-landing` | Copies des fichiers sources téléchargés ou préparés ; conservation de leur contenu original |
| `medical-warehouse` | Fichiers et métadonnées des tables Iceberg Bronze, Silver et Gold |


---
# PARTIE 0 — Préparer l'environnement et les données

**Objectif :** confirmer que les services fonctionnent et préparer les trois sources à ingérer. À la fin de cette partie, aucune table Bronze, Silver ou Gold ne doit encore être créée.

## 0.1 — Vérifier les services Docker

**Consignes**

1. Démarrer **Docker Desktop** et ouvrir le terminal Windows ou WSL.
2. Se placer dans le dossier du projet contenant `compose.yaml`.
3. Démarrer les services avec Docker Compose si ce n'est pas déjà fait.(docker compose config --quiet
docker compose up -d
docker compose ps)
4. Contrôler les états des conteneurs `medical-postgres`, `medical-rustfs`, `medical-iceberg-rest` et `medical-spark-jupyter`.
5. Ouvrir la console RustFS, l'API REST Iceberg et Jupyter via les adresses indiquées plus haut.
6. Vérifier la présence des buckets `medical-landing` et `medical-warehouse`.



## 0.2 — Télécharger le dataset officiel Synthea

Synthea est un logiciel libre de génération de **données de patients synthétiques** publié par l'équipe Synthea / MITRE.

**Téléchargement :**

- [Page officielle Synthea — Downloads](https://synthetichealth.github.io/downloads.html)
- [Archive officielle — 100 Sample Synthetic Patient Records (CSV)](https://synthetichealth.github.io/synthea-sample-data/downloads/latest/synthea_sample_data_csv_latest.zip)

**Consignes**

1. Télécharger l'archive CSV **des dossiers patients synthétiques** depuis le site officiel.
2. Extraire l'archive dans le dossier local du projet : `notebooks/data/synthea/`.
3. Vérifier que les fichiers `patients.csv`, `encounters.csv` et `observations.csv` sont présents (ils peuvent se trouver dans un sous-dossier selon l'organisation de l'archive).




## 0.3 — Démarrer Spark et configurer Iceberg

Nous allons travailler directement dans **Jupyter Notebook**, au sein de l'environnement Docker du projet.

Apache Spark permettra de lire les données, d'effectuer les traitements et de créer les tables du **Data Lakehouse**.

La session Spark sera configurée pour communiquer avec le catalogue **Apache Iceberg** et le stockage **RustFS**.

**Consignes :**

1. Exécuter le code suivant pour initialiser Spark.
2. Vérifier que la version de Spark s'affiche correctement.

**Code de configuration :**

```python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.medical",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.medical.type", "rest")
    .config(
        "spark.sql.catalog.medical.uri",
        "http://iceberg-rest:8181"
    )
    .config(
        "spark.sql.catalog.medical.warehouse",
        "s3://medical-warehouse/"
    )
    .config(
        "spark.sql.catalog.medical.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.medical.s3.endpoint",
        "http://rustfs:9000"
    )
    .config(
        "spark.sql.catalog.medical.s3.path-style-access",
        "true"
    )
    .getOrCreate()
)

print("Version de Spark :", spark.version)
```

**Résultat attendu :** la session Spark démarre et affiche sa version.

## 0.4 — Préparer les trois sources indépendantes

Le téléchargement fournit des CSV. Pour **simuler plusieurs systèmes hospitaliers**, nous réutilisons le même jeu cohérent, mais sous trois supports techniques distincts.

| Source à préparer | Fichier d'origine | Travail demandé | Source finale |
|---|---|---|---|
| Patients | `patients.csv` | Sélectionner les champs requis et charger une table SQL | PostgreSQL : `hospital.public.patients_source` |
| Rencontres | `encounters.csv` | Conserver le fichier CSV d'origine et repérer ses colonnes | Fichier CSV consultable par Spark |
| Observations | `observations.csv` | Convertir chaque ligne CSV en **un objet JSON par ligne** sans altérer les valeurs | `observations.jsonl` |

### Préparation de la table `patients_source`

Créer une table relationnelle avec la structure minimale suivante :

| Colonne PostgreSQL | Champ d'origine Synthea | Type pour la source du lab |
|---|---|---|
| `patient_id` | `Id` / `ID` | `TEXT`, clé unique |
| `birthdate` | `BIRTHDATE` | `TEXT` au chargement |
| `gender` | `GENDER` | `TEXT` |
| `city` | `CITY` | `TEXT` |

**Consignes :**

1. Lire le fichier `patients.csv` avec Spark.
2. Sélectionner les colonnes `Id`, `BIRTHDATE`, `GENDER` et `CITY`.
3. Créer et alimenter la table `patients_source` dans la base PostgreSQL `hospital`.
4. Vérifier le nombre de patients importés.




#### Préparation du fichier `observations.jsonl`
**Consignes :**

1. Lire le fichier `observations.csv` avec Python.
2. Convertir les données en fichier JSON Lines (`observations.jsonl`), avec un objet JSON par ligne.
3. Enregistrer le fichier dans `notebooks/data/prepared/`.
4. Vérifier que les observations ont été correctement converties.

**Pourquoi JSON Lines ?** Spark peut le lire comme une collection de documents semi-structurés. La transformation ici sert seulement à simuler une source de laboratoire au format JSON .

---
# PARTIE 1 — BRONZE : ingestion des données

**Objectif :** créer trois tables Iceberg qui représentent les données ingérées.

**Travail à réaliser**

1. Créer le namespace Iceberg `medical.bronze`.
2. Lire la table `patients_source` de PostgreSQL avec Spark JDBC.
3. Lire `encounters.csv` dans Spark.
4. Lire `observations.jsonl` dans Spark.
5. Afficher les schémas et les effectifs avant ingestion.
6. Créer et alimenter les tables :

| Table attendue | Origine |
|---|---|
| `medical.bronze.patients` | PostgreSQL `patients_source` |
| `medical.bronze.consultations` | CSV `encounters.csv` |
| `medical.bronze.observations` | JSON Lines `observations.jsonl` |


---
# PARTIE 2 — SILVER : nettoyage, cohérence et qualité

**Objectif :** transformer les données Bronze en trois tables propres, sans perdre la traçabilité des anomalies. Toutes les règles doivent être explicites et justifiées par le format des données, non par une interprétation médicale non validée.

## 2.1 — Nettoyer les patients et les rencontres

**Consignes**

1. Lire `medical.bronze.patients` et `medical.bronze.consultations`.
2. Harmoniser les noms des colonnes utiles et les types (dates, identifiants, champs textuels).
3. Vérifier les identifiants indispensables, les doublons et la cohérence des dates.
4. Vérifier la correspondance entre l'identifiant patient des rencontres et la table des patients.
5. Enregistrer les données valides dans `medical.silver.patients` et `medical.silver.consultations`.
6. Compter séparément les lignes d'origine, valides et non conformes.

**Attention :** ne pas supprimer silencieusement les lignes problématiques ; elles doivent pouvoir être analysées dans Quarantine.(partie suivante)

## 2.2 — Normaliser les observations médicales

**Consignes**

1. Lire `medical.bronze.observations`.
2. Conserver les identifiants de patient et, s'il existe, l'identifiant de rencontre.
3. Identifier les champs de date, code, description, valeur et unité présents dans le schéma réel.
4. Normaliser uniquement les types vérifiables, les dates et les espaces inutiles.
5. Vérifier les références vers les patients et les rencontres quand les colonnes pertinentes sont disponibles.
6. Alimenter `medical.silver.observations` avec les enregistrements valides.

> **Remarque :** les observations Synthea peuvent contenir des résultats numériques **ou textuels**. Ne pas convertir aveuglément toutes les valeurs en nombres.

## 2.3 — Mettre en quarantaine les anomalies

**Consignes**

1. Définir des règles simples pour détecter les données incorrectes (valeurs nulles, identifiants manquants, dates invalides).
2. Créer le namespace `medical.quarantine`.
3. Stocker les données incorrectes dans table Iceberg, en indiquant le motif du rejet.

**Résultat attendu :** une table Quarantine contenant les données rejetées et les raisons de leur rejet.



---
# PARTIE 3 — GOLD : indicateurs d'activité hospitalière

**Objectif :** créer des tables synthétiques pertinentes à partir de **Silver**, destinées à l'analyse SQL et à de futurs tableaux de bord.

## 3.1 — Construire deux tables analytiques Gold

**Objectif :** utiliser les données Silver pour produire des indicateurs hospitaliers à l'aide de Spark SQL.

| Table Iceberg Gold | Sources Silver | Indicateur à calculer |
|---|---|---|
| `medical.gold.rencontres_par_classe_et_sexe` | `silver.consultations` + `silver.patients` | Nombre de rencontres médicales par catégorie (`categorie`) et par genre (`gender`) |
| `medical.gold.observations_par_type` | `silver.observations` | Nombre d'observations médicales par code ou description |

### Consignes

1. Créer le namespace `medical.gold`.
2. Effectuer une jointure entre `medical.silver.consultations` et `medical.silver.patients` à l'aide de `patient_id`.
3. Calculer le nombre de rencontres par catégorie et par genre, puis enregistrer les résultats dans `medical.gold.rencontres_par_classe_et_sexe`.
4. Calculer le nombre d'observations médicales par code ou description, puis enregistrer les résultats dans `medical.gold.observations_par_type`.
5. Afficher le contenu des deux tables avec Spark SQL.

**Résultat attendu :** deux tables analytiques Gold contenant les indicateurs calculés à partir des données Silver, consultables avec Spark SQ


## 3.2 — Analyser les données hospitalières avec Spark SQL

**Objectif :** exploiter les tables Silver et Gold pour produire des indicateurs sur l'activité hospitalière.

### Consignes

**1. Réaliser les quatre analyses suivantes avec Spark SQL :**

| Analyse | Données à utiliser |
|---|---|
| Nombre de consultations par catégorie | Table Gold des rencontres |
| Évolution mensuelle des consultations | `medical.silver.consultations` |
| Nombre d'observations par type | `medical.gold.observations_par_type` |
| Répartition des patients par genre | `medical.silver.patients` |

**2. Vérifier les résultats :**

1. Afficher les résultats de chaque analyse avec Spark SQL.
2. Trier les résultats par nombre de consultations, d'observations ou de patients, selon l'indicateur.
3. Vérifier la cohérence des résultats avec les données Silver.


## PARTIE 4 — Fonctionnalités avancées d'Apache Iceberg (facultatif)

**Objectif :** découvrir les snapshots et le Time Travel d'Apache Iceberg.

### 4.1 — Snapshots et Time Travel

**Consignes**

1. Créer une table Iceberg `medical.sandbox.demo_versioning` contenant deux patients fictifs : `P001` et `P002`.
2. Afficher les données et consulter les snapshots avec Spark SQL.
3. Ajouter un troisième patient `P003`.
4. Vérifier qu'un nouveau snapshot a été créé.
5. Utiliser `VERSION AS OF` pour consulter la version précédente contenant uniquement deux patients.


##Bon travail .
